In [1]:
import io
import json
import hashlib
import re
import time
import traceback
from pathlib import Path
from typing import Any, Optional

import ollama
import pypdfium2 as pdfium

from pydantic import BaseModel, Field, ConfigDict


# ============================================================
# CONFIGURATION
# ============================================================

INPUT_DIR = Path(r"G:\Projects\Instant-ODC\data\raw")
OUTPUT_DIR = Path(r"G:\Projects\Instant-ODC\data\processed")

EXTRACTIONS_DIR = OUTPUT_DIR / "extractions"
RAW_RESPONSES_DIR = OUTPUT_DIR / "raw_responses"
CHECKPOINTS_DIR = OUTPUT_DIR / "checkpoints"
REPORTS_DIR = OUTPUT_DIR / "reports"
EMBEDDINGS_DIR = OUTPUT_DIR / "embeddings"

for directory in (
    OUTPUT_DIR,
    EXTRACTIONS_DIR,
    RAW_RESPONSES_DIR,
    CHECKPOINTS_DIR,
    REPORTS_DIR,
    EMBEDDINGS_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)


# ============================================================
# MODELS
# ============================================================

QWEN_MODEL = "qwen2.5vl:3b"
EMBED_MODEL = "embeddinggemma"


# ============================================================
# PDF PROCESSING
# ============================================================

PDF_EXTENSIONS = {".pdf"}
RENDER_SCALE = 2.0
JPEG_QUALITY = 90


# ============================================================
# NATIVE TEXT EXTRACTION GATES
# ============================================================

MIN_NATIVE_TEXT_CHARS = 150
MIN_NATIVE_WORDS = 25
MAX_GARBAGE_RATIO = 0.05
MAX_REPLACEMENT_CHAR_RATIO = 0.01


# ============================================================
# QUALITY
# ============================================================

MIN_EXTRACTED_CHARACTERS = 50
SUSPICIOUSLY_LOW_CHARACTERS = 150


# ============================================================
# STARTING PAGE
# ============================================================


START_PAGE = 1
REPROCESS_NATIVE = False

# ============================================================
# SPEED KNOBS
# ============================================================

KEEP_ALIVE = "1h"
OLLAMA_TIMEOUT_SECONDS = 1800
EMBED_BATCH_SIZE = 32
EMBED_MAX_CHARS = 6000


RETRY_CONFIGS = [
    {"name": "attempt_1", "render_scale": 2.0, "num_ctx": 16384, "num_predict": 8192},
    {"name": "attempt_2", "render_scale": 2.0, "num_ctx": 16384, "num_predict": 16384},
    {"name": "attempt_3", "render_scale": 3.0, "num_ctx": 16384, "num_predict": 16384},
    {"name": "attempt_4", "render_scale": 2.0, "num_ctx": 16384, "num_predict": 24576},
]

_RETRY_BY_NAME = {c["name"]: c for c in RETRY_CONFIGS}


def _next_plan(failed_name: str, done_reason: Optional[str]) -> list[dict]:
    """Pick the next attempt based on HOW the previous one failed."""
    length_cut = done_reason == "length"
    if failed_name == "attempt_1":
        # Output was cut -> more tokens. Otherwise -> sharper image.
        return [_RETRY_BY_NAME["attempt_2"]] if length_cut else [_RETRY_BY_NAME["attempt_3"]]
    if failed_name == "attempt_2":
        # Still truncating at 16k output -> max output budget.
        return [_RETRY_BY_NAME["attempt_4"]]
    if failed_name == "attempt_3":
        return [_RETRY_BY_NAME["attempt_4"]] if length_cut else []
    return []



QWEN_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "page_number": {"type": "integer"},
        "title": {"type": ["string", "null"]},
        "sections": {"type": "array", "items": {"type": "string"}},
        "blocks": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "type": {"type": "string"},
                    "text": {"type": "string"},
                },
                "required": ["type", "text"],
            },
        },
        "tables": {"type": "array", "items": {"type": "string"}},
        "figures": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["page_number", "sections", "blocks", "tables", "figures"],
}


# ============================================================
# OLLAMA CLIENT (with timeout, graceful fallback)
# ============================================================

def _make_client():
    try:
        return ollama.Client(timeout=OLLAMA_TIMEOUT_SECONDS)
    except TypeError:
        return ollama  # module-level functions (older ollama-python)


CLIENT = _make_client()
_json_schema_supported = {"state": None}


# ============================================================
# WORD-EXTRACTION RULES 
# ============================================================

_LIGATURES = str.maketrans({
    "\ufb00": "ff",
    "\ufb01": "fi",
    "\ufb02": "fl",
    "\ufb03": "ffi",
    "\ufb04": "ffl",
})

_INVISIBLES = dict.fromkeys(map(ord, "\u200b\u200c\u200d\u2060\ufeff"), None)

_SPACES_RE = re.compile(r"[\u00a0\u1680\u2000-\u200a\u202f\u205f\u3000]")

_CTRL_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")

_MULTI_SPACE_RE = re.compile(r"[ \t]{2,}")

_WORD_RE = re.compile(r"[^\W_]+", re.UNICODE)

_NUM_RE = re.compile(r"\d+(?:[.,]\d+)?")

_PAGE_NUM_RE = re.compile(r"^(?:page\s+)?\d{1,4}(?:\s+of\s+\d{1,4})?$", re.IGNORECASE)

_BULLET_RE = re.compile(r"^[\u2022\u2023\u25aa\u25cf\u25e6\-*\u00b7>]\s+")

_LABEL_RE = re.compile(r"^[A-Za-z][A-Za-z ()/_-]{0,40}:")

_DATA_RE = re.compile(
    r"\b\d+(?:[.,]\d+)?\s*(mg|mcg|g|kg|mL|L|%|mmol|mmHg|IU|U|mm|cm)\b",
    re.IGNORECASE,
)


def sanitize_text(text: str) -> str:
    """Unicode hygiene. Preserves \n, \r, \t and all medical symbols."""
    if not text:
        return ""
    text = text.translate(_LIGATURES)     
    text = text.replace("\u00ad", "")   
    text = text.translate(_INVISIBLES)     
    text = _SPACES_RE.sub(" ", text)      
    text = _CTRL_RE.sub(" ", text)        
    return _MULTI_SPACE_RE.sub(" ", text)


def join_paragraph(lines: list[str]) -> str:
    """
    Join lines into a paragraph, merging words hyphen-split across
    line breaks:  "fail-" + "ure" -> "failure".
    Only merges when the next line starts with a lowercase letter
    (the standard dehyphenation rule for justified text).
    """
    parts: list[str] = []
    for line in lines:
        if (
            parts
            and parts[-1].endswith("-")
            and len(parts[-1]) >= 3
            and parts[-1][-2].isalpha()   
            and line[:1].islower()        
        ):
            parts[-1] = parts[-1][:-1] + line
        else:
            parts.append(line)
    return " ".join(parts)


def looks_like_heading(line: str) -> bool:
    """
    Conservative heading detection. Deliberately biased toward
    "paragraph" so medication lists, field labels, and lab values
    are never fragmented into fake sections.
    """
    if not (5 <= len(line) <= 120):
        return False
    words = line.split()
    if not 1 <= len(words) <= 12:
        return False
    if _BULLET_RE.match(line):                    
        return False
    if _LABEL_RE.match(line):                      
        return False
    if _DATA_RE.search(line):                 
        return False
    if line.endswith((".", ",", ";", ":", "!", "?")):
        return False
    if line.isupper() and any(c.isalpha() for c in line):
        return True                             
    cased = [w for w in words if any(c.isalpha() for c in w)]
    if not cased:
        return False                              
    return sum(1 for w in cased if w[0].isupper()) / len(cased) >= 0.6


# ============================================================
# PYDANTIC SCHEMAS
# ============================================================

class QwenBlock(BaseModel):
    model_config = ConfigDict(extra="allow")

    type: str = "paragraph"
    text: Any = None
    content: Any = None

    def normalized_text(self) -> str:
        value = self.text
        if value is None:
            value = self.content
        if value is None:
            return ""
        if isinstance(value, str):
            return value
        if isinstance(value, (dict, list)):
            return json.dumps(value, ensure_ascii=False)
        return str(value)


class QwenExtractionResult(BaseModel):
    model_config = ConfigDict(extra="allow")

    page_number: int
    title: Optional[str] = None
    sections: list[Any] = Field(default_factory=list)
    blocks: list[QwenBlock] = Field(default_factory=list)
    tables: list[Any] = Field(default_factory=list)
    figures: list[Any] = Field(default_factory=list)


class ExtractedBlock(BaseModel):
    type: str
    text: str


class PageExtractionResult(BaseModel):
    page_number: int
    title: Optional[str] = None
    sections: list[str] = Field(default_factory=list)
    blocks: list[ExtractedBlock] = Field(default_factory=list)
    tables: list[str] = Field(default_factory=list)
    figures: list[str] = Field(default_factory=list)


class PageCheckpoint(BaseModel):
    pdf_name: str
    pdf_hash: str
    page_number: int
    status: str                  
    method: str = "unknown"
    attempts: int = 0
    successful_attempt: Optional[int] = None
    done_reason: Optional[str] = None
    time_seconds: float = 0.0
    model_time_seconds: float = 0.0
    sections: int = 0
    blocks: int = 0
    tables: int = 0
    figures: int = 0
    extracted_characters: int = 0
    quality_status: str = "unknown"
    quality_reasons: list[str] = Field(default_factory=list)
    extraction_path: Optional[str] = None
    raw_response_paths: list[str] = Field(default_factory=list)
    error_type: Optional[str] = None
    error: Optional[str] = None
    first_line: Optional[str] = None  


COMPLETE_STATUSES = {"success", "review", "empty_page"}


# ============================================================
# UTILITIES
# ============================================================

_RESERVED_NAMES = {
    "CON", "PRN", "AUX", "NUL",
    *(f"COM{i}" for i in range(1, 10)),
    *(f"LPT{i}" for i in range(1, 10)),
}


def safe_filename(name: str) -> str:
    invalid = '<>:"/\\|?*'
    result = "".join("_" if c in invalid else c for c in name)
    result = result.strip().rstrip(". ")
    if not result:
        result = "unnamed"
    if result.upper() in _RESERVED_NAMES:
        result = "_" + result
    return result


_hash_cache: dict[str, tuple[int, float, str]] = {}


def pdf_identifier(pdf_path: Path) -> str:
    key = str(pdf_path)
    try:
        stat = pdf_path.stat()
        cached = _hash_cache.get(key)
        if cached and cached[0] == stat.st_size and cached[1] == stat.st_mtime:
            return cached[2]
    except OSError:
        pass

    sha256 = hashlib.sha256()
    with open(pdf_path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            sha256.update(chunk)
    digest = sha256.hexdigest()
    _hash_cache[key] = (stat.st_size, stat.st_mtime, digest)
    return digest


def get_pdf_output_dirs(pdf_path: Path):
    pdf_name = safe_filename(pdf_path.stem)
    extraction_dir = EXTRACTIONS_DIR / pdf_name
    raw_dir = RAW_RESPONSES_DIR / pdf_name
    checkpoint_dir = CHECKPOINTS_DIR / pdf_name
    embedding_dir = EMBEDDINGS_DIR / pdf_name

    for directory in (extraction_dir, raw_dir, checkpoint_dir, embedding_dir):
        directory.mkdir(parents=True, exist_ok=True)

    return pdf_name, extraction_dir, raw_dir, checkpoint_dir, embedding_dir


# ============================================================
# OLLAMA CHECK
# ============================================================

def test_ollama_connection() -> bool:
    print("\n" + "=" * 70)
    print("OLLAMA CONNECTION")
    print("=" * 70)
    try:
        models = CLIENT.list()
        installed = [m.model for m in models.models]
        print("\nInstalled models:")
        for model in installed:
            print(f"  - {model}")

        if not any(QWEN_MODEL in name for name in installed):
            print(f"\n✗ Missing: {QWEN_MODEL}  (run: ollama pull {QWEN_MODEL})")
            return False
        if not any(EMBED_MODEL in name for name in installed):
            print(f"\n✗ Missing: {EMBED_MODEL}  (run: ollama pull {EMBED_MODEL})")
            return False

        print("\n✓ Ollama is reachable.")
        print("✓ Required models are installed.")
        return True
    except Exception as e:
        print(f"\n✗ Ollama connection failed: {type(e).__name__}: {e}")
        return False


# ============================================================
# NATIVE TEXT EXTRACTION (pypdfium2, Phase 1)
# ============================================================

def extract_native_text(pdf, page_number: int) -> str:
    """Return RAW page text (quality gates measure the raw text;
    sanitization happens in native_text_to_result)."""
    page = pdf[page_number - 1]
    try:
        text_page = page.get_textpage()
        try:
            return text_page.get_text_range() or ""
        finally:
            text_page.close()
    finally:
        page.close()


def calculate_garbage_ratio(text: str) -> float:
    if not text:
        return 1.0
    garbage = 0
    for char in text:
        if char == "\ufffd":
            garbage += 1
        elif ord(char) < 32 and char not in "\n\r\t":
            garbage += 1
    return garbage / len(text)


def native_text_quality(text: str) -> tuple[bool, list[str]]:
    reasons = []
    stripped = text.strip()
    if not stripped:
        return False, ["empty_native_text"]

    char_count = len(stripped)
    word_count = len(_WORD_RE.findall(stripped))       # real words only
    garbage_ratio = calculate_garbage_ratio(stripped)
    replacement_ratio = stripped.count("\ufffd") / max(len(stripped), 1)

    if char_count < MIN_NATIVE_TEXT_CHARS:
        reasons.append(f"too_few_characters_{char_count}")
    if word_count < MIN_NATIVE_WORDS:
        reasons.append(f"too_few_words_{word_count}")
    if garbage_ratio > MAX_GARBAGE_RATIO:
        reasons.append(f"high_garbage_ratio_{garbage_ratio:.3f}")
    if replacement_ratio > MAX_REPLACEMENT_CHAR_RATIO:
        reasons.append(f"high_replacement_ratio_{replacement_ratio:.3f}")

    return len(reasons) == 0, reasons


def native_text_to_result(
    page_number: int,
    text: str,
    prev_first_line: Optional[str] = None,
    is_first_page: bool = False,
) -> tuple[PageExtractionResult, Optional[str]]:
    """
    Build a structured result from native text.
    Returns (result, raw_first_line_for_next_page).

    Rules applied:
      - Unicode sanitization (ligatures, soft hyphens, spaces)
      - Running-header removal (identical first line on consecutive pages)
      - Footer page-number removal
      - Dehyphenation across line breaks
      - Conservative heading detection
      - Title only on page 1 (never a running header)
    """
    text = sanitize_text(text)
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]

    if not lines:
        return PageExtractionResult(page_number=page_number), None

    raw_first_line = lines[0]

    # Running header: identical first line as the previous page.
    if prev_first_line and lines[0] == prev_first_line:
        lines = lines[1:]

    # Footer page number: "12", "Page 12", "Page 12 of 340".
    if lines and _PAGE_NUM_RE.fullmatch(lines[-1]):
        lines = lines[:-1]

    sections: list[str] = []
    blocks: list[ExtractedBlock] = []
    paragraph: list[str] = []

    def flush_paragraph():
        if paragraph:
            para = join_paragraph(paragraph)
            if para:
                blocks.append(ExtractedBlock(type="paragraph", text=para))
            paragraph.clear()

    for line in lines:
        if looks_like_heading(line):
            flush_paragraph()
            sections.append(line)
        else:
            paragraph.append(line)

    flush_paragraph()

    # Title only from page 1 -> never a running header.
    title = sections[0] if (is_first_page and sections) else None

    result = PageExtractionResult(
        page_number=page_number,
        title=title,
        sections=sections,
        blocks=blocks,
        tables=[],
        figures=[],
    )
    return result, raw_first_line


# ============================================================
# RENDER PAGE IN MEMORY (JPEG, Phase 2)
# ============================================================

def render_page_to_memory(pdf, page_number: int, scale: float = RENDER_SCALE) -> bytes:
    page = pdf[page_number - 1]
    try:
        bitmap = page.render(scale=scale)
        try:
            image = bitmap.to_pil()
            buffer = io.BytesIO()
            image.save(buffer, format="JPEG", quality=JPEG_QUALITY)
            image.close()
            return buffer.getvalue()
        finally:
            close = getattr(bitmap, "close", None)
            if close:
                close()
    finally:
        page.close()


# ============================================================
# QWEN PROMPT
# ============================================================

def build_prompt(page_number: int, total_pages: int) -> str:
    return f"""You are extracting page {page_number} of {total_pages}
from a medical document.

Perform SOURCE-FAITHFUL DOCUMENT EXTRACTION.
Read the entire visible page.

CRITICAL RULES:
1. Extract only information visibly present. Never invent information.
2. Never infer missing medical facts. Never medically correct the source.
3. Do not summarize. Extract the actual visible content.
4. Preserve wording, reading order, headings and footnotes as closely as possible.
5. Preserve EXACTLY, character for character: numbers, decimal values,
   percentages, units, medication names, medication doses, laboratory
   values, medical abbreviations, dates, ranges, and comparison symbols.
6. Tables: represent the visible structure. Do not invent missing cells.
7. Figures/charts: describe only what is visibly present.
8. If something is unclear, preserve only what can actually be read.

Identify: title, section headings, paragraphs, lists, important text
blocks, tables, figures, charts, captions, references.

Return ONLY valid JSON using exactly this shape:

{{
    "page_number": {page_number},
    "title": null,
    "sections": [],
    "blocks": [
        {{
            "type": "paragraph",
            "text": "..."
        }}
    ],
    "tables": [],
    "figures": []
}}

Blocks must use "text" (a string). Do not use Markdown."""


# ============================================================
# NORMALIZATION
# ============================================================

def stringify_value(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, str):
        return value
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return str(value)


def normalize_result(qwen_result: QwenExtractionResult) -> PageExtractionResult:
    sections = []
    for section in qwen_result.sections:
        value = sanitize_text(stringify_value(section)).strip()
        if value:
            sections.append(value)

    blocks = []
    tables = []
    figures = []

    for block in qwen_result.blocks:
        text = sanitize_text(block.normalized_text()).strip()
        if not text:
            continue
        block_type = (block.type or "paragraph").strip()
        if block_type.lower() in {"table", "tabular"}:
            tables.append(text)
        else:
            blocks.append(ExtractedBlock(type=block_type, text=text))

    for table in qwen_result.tables:
        value = sanitize_text(stringify_value(table)).strip()
        if value:
            tables.append(value)

    for figure in qwen_result.figures:
        value = sanitize_text(stringify_value(figure)).strip()
        if value:
            figures.append(value)

    return PageExtractionResult(
        page_number=qwen_result.page_number,
        title=qwen_result.title,
        sections=sections,
        blocks=blocks,
        tables=tables,
        figures=figures,
    )


# ============================================================
# CONTENT QUALITY
# ============================================================

def calculate_extracted_characters(result: PageExtractionResult) -> int:
    total = 0
    if result.title:
        total += len(result.title)
    for section in result.sections:
        total += len(section)
    for block in result.blocks:
        total += len(block.text)
    for table in result.tables:
        total += len(table)
    for figure in result.figures:
        total += len(figure)
    return total


def assess_quality(result: PageExtractionResult) -> tuple[str, list[str]]:
    characters = calculate_extracted_characters(result)
    objects = (
        len(result.sections)
        + len(result.blocks)
        + len(result.tables)
        + len(result.figures)
    )
    reasons = []

    if objects == 0:
        return "needs_reextraction", ["no_extracted_content"]
    if characters == 0:
        return "needs_reextraction", ["zero_extracted_characters"]
    if characters < MIN_EXTRACTED_CHARACTERS:
        return "needs_reextraction", [f"very_low_content_{characters}_characters"]
    if characters < SUSPICIOUSLY_LOW_CHARACTERS:
        return "review", [f"low_content_{characters}_characters"]
    return "pass", reasons


def build_embedding_text(result: PageExtractionResult) -> str:
    pieces = []
    if result.title:
        pieces.append(result.title)
    pieces.extend(result.sections)
    for block in result.blocks:
        pieces.append(block.text)
    pieces.extend(result.tables)
    pieces.extend(result.figures)
    return "\n\n".join(x for x in pieces if x)


def numeric_cross_check(
    page_number: int,
    native_text: str,
    native_reasons: list[str],
    result: PageExtractionResult,
) -> list[str]:
    """
    Free precision check: numbers present in the native text layer but
    missing from the Qwen extraction are a hallucination/drop red flag.
    Only downgrades pass -> review. Never triggers retries.
    """
    if not native_text or len(native_text.strip()) < 50:
        return []
    # Unreliable native text (encoding garbage) -> skip the check.
    if any("garbage" in r or "replacement" in r for r in native_reasons):
        return []

    native_nums = set(_NUM_RE.findall(native_text))
    native_nums.discard(str(page_number))  # page number in header/footer
    if not native_nums:
        return []

    extracted_nums = set(_NUM_RE.findall(build_embedding_text(result)))
    missing = sorted(native_nums - extracted_nums, key=lambda x: (len(x), x))
    if not missing:
        return []

    shown = ", ".join(missing[:10])
    return [f"numbers_missing_vs_native_text[{shown}]"]


# ============================================================
# CHECKPOINTS
# ============================================================

def checkpoint_path(checkpoint_dir: Path, page_number: int) -> Path:
    return checkpoint_dir / f"page_{page_number:04d}.json"


def save_checkpoint(checkpoint_dir: Path, checkpoint: PageCheckpoint):
    path = checkpoint_path(checkpoint_dir, checkpoint.page_number)
    temp_path = path.with_suffix(".tmp")
    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(checkpoint.model_dump(), f, indent=2, ensure_ascii=False)
    temp_path.replace(path)


def load_checkpoint(checkpoint_dir: Path, page_number: int) -> Optional[PageCheckpoint]:
    path = checkpoint_path(checkpoint_dir, page_number)
    if not path.exists():
        return None
    try:
        with open(path, "r", encoding="utf-8") as f:
            return PageCheckpoint.model_validate(json.load(f))
    except Exception:
        return None


# ============================================================
# EXTRACTION FILE / RAW RESPONSE
# ============================================================

def save_extraction(extraction_dir: Path, result: PageExtractionResult) -> Path:
    path = extraction_dir / f"page_{result.page_number:04d}.json"
    temp_path = path.with_suffix(".tmp")
    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(result.model_dump(), f, indent=2, ensure_ascii=False)
    temp_path.replace(path)
    return path


def save_raw_response(
    raw_dir: Path, page_number: int, attempt_number: int, raw_response: str
) -> Path:
    path = raw_dir / f"page_{page_number:04d}_attempt_{attempt_number}.txt"
    with open(path, "w", encoding="utf-8") as f:
        f.write(raw_response)
    return path


# ============================================================
# QWEN ATTEMPT (schema-constrained, keep_alive, timeout)
# ============================================================

def run_qwen_attempt(
    image_bytes: bytes,
    page_number: int,
    total_pages: int,
    config: dict,
) -> dict:
    prompt = build_prompt(page_number, total_pages)
    messages = [{"role": "user", "content": prompt, "images": [image_bytes]}]
    options = {
        "temperature": 0,
        "num_ctx": config["num_ctx"],
        "num_predict": config["num_predict"],
    }

    use_schema = _json_schema_supported["state"] != "json"
    fmt = QWEN_JSON_SCHEMA if use_schema else "json"

    start = time.perf_counter()
    try:
        response = CLIENT.chat(
            model=QWEN_MODEL,
            messages=messages,
            format=fmt,
            options=options,
            keep_alive=KEEP_ALIVE,
        )
        if use_schema and _json_schema_supported["state"] is None:
            _json_schema_supported["state"] = "schema"
    except Exception as e:
        # Older server/client can't do structured output -> fall back once.
        if not use_schema:
            raise
        msg = str(e).lower()
        if any(k in msg for k in ("format", "schema", "json", "400", "422")):
            _json_schema_supported["state"] = "json"
            response = CLIENT.chat(
                model=QWEN_MODEL,
                messages=messages,
                format="json",
                options=options,
                keep_alive=KEEP_ALIVE,
            )
        else:
            raise

    model_time = time.perf_counter() - start
    raw_response = response.message.content or ""
    done_reason = getattr(response, "done_reason", None)
    return {
        "raw_response": raw_response,
        "done_reason": done_reason,
        "model_time": model_time,
    }


# ============================================================
# QWEN PAGE PROCESSING (Phase 2)
# ============================================================

def process_qwen_page(
    pdf,
    pdf_name: str,
    pdf_hash: str,
    page_number: int,
    total_pages: int,
    extraction_dir: Path,
    raw_dir: Path,
    checkpoint_dir: Path,
    native_text: str = "",
    native_reasons: Optional[list[str]] = None,
) -> PageCheckpoint:
    overall_start = time.perf_counter()
    native_text = native_text or ""
    native_reasons = native_reasons or []
    first_line = next(
        (ln.strip() for ln in native_text.splitlines() if ln.strip()), None
    )

    raw_paths: list[str] = []
    total_model_time = 0.0
    last_error_type: Optional[str] = None
    last_error: Optional[str] = None
    last_done_reason: Optional[str] = None
    saw_valid_empty = False   # model saw the page and legitimately found nothing
    attempts_made = 0
    image_cache: dict[float, bytes] = {}

    plan: list[dict] = [RETRY_CONFIGS[0]]

    try:
        while plan:
            config = plan.pop(0)
            attempts_made += 1
            scale = config["render_scale"]

            # Render once per scale (attempts 1/2 share scale 2.0).
            if scale not in image_cache:
                print(f"  → Rendering page (scale {scale})...")
                image_cache[scale] = render_page_to_memory(pdf, page_number, scale)
                print(f"  ✓ Rendered ({len(image_cache[scale]) / 1024:.1f} KB)")

            print(
                f"  → Qwen attempt {attempts_made} "
                f"(ctx={config['num_ctx']}, output={config['num_predict']}, "
                f"scale={scale})"
            )

            try:
                response = run_qwen_attempt(
                    image_bytes=image_cache[scale],
                    page_number=page_number,
                    total_pages=total_pages,
                    config=config,
                )
            except Exception as e:
                last_error_type = type(e).__name__
                last_error = str(e)
                print(f"  ⚠ Attempt error: {last_error_type}: {last_error}")
                err = str(e).lower()
                if "memory" in err or "oom" in err:
                    plan = []  # escalating scale/ctx would only make OOM worse
                else:
                    plan = _next_plan(config["name"], None)
                continue

            raw = response["raw_response"]
            done_reason = response["done_reason"]
            total_model_time += response["model_time"]
            last_done_reason = done_reason

            raw_path = save_raw_response(raw_dir, page_number, attempts_made, raw)
            raw_paths.append(str(raw_path))

            # --- Parse (grammar-constrained output should never fail here)
            try:
                parsed = json.loads(raw)
            except json.JSONDecodeError as e:
                last_error_type = "JSONDecodeError"
                last_error = str(e)
                print("  ⚠ Invalid JSON.")
                plan = _next_plan(config["name"], done_reason)
                continue

            # We know the page number; never trust the model's echo of it.
            if isinstance(parsed, dict):
                parsed["page_number"] = page_number

            try:
                qwen_result = QwenExtractionResult.model_validate(parsed)
            except Exception as e:
                last_error_type = "ValidationError"
                last_error = str(e)
                print("  ⚠ Schema validation failed.")
                plan = _next_plan(config["name"], done_reason)
                continue

            try:
                final_result = normalize_result(qwen_result)
            except Exception as e:
                last_error_type = "NormalizationError"
                last_error = str(e)
                print("  ⚠ Normalization failed.")
                plan = _next_plan(config["name"], done_reason)
                continue

            quality_status, quality_reasons = assess_quality(final_result)
            characters = calculate_extracted_characters(final_result)

            if quality_status == "needs_reextraction":
                if any(
                    r in ("no_extracted_content", "zero_extracted_characters")
                    or r.startswith("very_low_content")
                    for r in quality_reasons
                ):
                    saw_valid_empty = True
                last_error_type = "QualityFailure"
                last_error = "; ".join(quality_reasons)
                print("  ⚠ Qwen extraction was too small.")
                plan = _next_plan(config["name"], done_reason)
                continue

            # Free precision check against the native text layer.
            if quality_status == "pass":
                cross = numeric_cross_check(
                    page_number, native_text, native_reasons, final_result
                )
                if cross:
                    quality_status = "review"
                    quality_reasons = quality_reasons + cross
                    print(f"  ⚠ Numeric cross-check flagged: {cross[0][:100]}")

            extraction_path = save_extraction(extraction_dir, final_result)
            total_time = time.perf_counter() - overall_start

            checkpoint = PageCheckpoint(
                pdf_name=pdf_name,
                pdf_hash=pdf_hash,
                page_number=page_number,
                status="success" if quality_status == "pass" else "review",
                method="qwen",
                attempts=attempts_made,
                successful_attempt=attempts_made,
                done_reason=done_reason,
                time_seconds=round(total_time, 2),
                model_time_seconds=round(total_model_time, 2),
                sections=len(final_result.sections),
                blocks=len(final_result.blocks),
                tables=len(final_result.tables),
                figures=len(final_result.figures),
                extracted_characters=characters,
                quality_status=quality_status,
                quality_reasons=quality_reasons,
                extraction_path=str(extraction_path),
                raw_response_paths=raw_paths,
                first_line=first_line,
            )
            save_checkpoint(checkpoint_dir, checkpoint)

            print(f"  ✓ Qwen extraction successful ({characters:,} chars).")
            if attempts_made > 1:
                print(f"  ✓ Recovered on attempt {attempts_made}.")
            return checkpoint

        # --- All attempts exhausted
        total_time = time.perf_counter() - overall_start

        # Genuinely blank page (model looked and found nothing, native layer
        # also empty) -> TERMINAL state. No more retries on future runs.
        if saw_valid_empty and len(native_text.strip()) < MIN_NATIVE_TEXT_CHARS:
            status = "empty_page"
            quality_status = "empty_page"
            quality_reasons = ["page_appears_blank"]
            print("  ✓ Page appears genuinely blank — marked terminal.")
        else:
            status = "needs_reextraction"
            quality_status = "needs_reextraction"
            quality_reasons = ["all_qwen_attempts_failed"]
            print(f"  ✗ Qwen failed after {attempts_made} attempt(s).")

        checkpoint = PageCheckpoint(
            pdf_name=pdf_name,
            pdf_hash=pdf_hash,
            page_number=page_number,
            status=status,
            method="qwen",
            attempts=attempts_made,
            successful_attempt=None,
            done_reason=last_done_reason,
            time_seconds=round(total_time, 2),
            model_time_seconds=round(total_model_time, 2),
            quality_status=quality_status,
            quality_reasons=quality_reasons,
            raw_response_paths=raw_paths,
            error_type=last_error_type,
            error=last_error,
            first_line=first_line,
        )
        save_checkpoint(checkpoint_dir, checkpoint)
        return checkpoint

    finally:
        image_cache.clear()


# ============================================================
# NATIVE PAGE ATTEMPT (Phase 1)
# ============================================================

def try_native_page(
    pdf,
    page_number: int,
    prev_first_line: Optional[str],
) -> tuple[Optional[PageExtractionResult], str, list[str], Optional[str]]:
    """
    Returns (result | None, native_text, failure_reasons, first_line).
    result is None when the page must go to the Qwen queue.
    """
    try:
        text = extract_native_text(pdf, page_number)
    except Exception as e:
        return None, "", [f"native_extraction_error_{type(e).__name__}"], None

    first_line = next((ln.strip() for ln in text.splitlines() if ln.strip()), None)

    good, reasons = native_text_quality(text)
    if not good:
        return None, text, reasons, first_line

    result, _ = native_text_to_result(
        page_number,
        text,
        prev_first_line=prev_first_line,
        is_first_page=(page_number == 1),
    )

    quality_status, quality_reasons = assess_quality(result)
    if quality_status == "needs_reextraction":
        return None, text, quality_reasons, first_line

    return result, text, [], first_line


# ============================================================
# SINGLE PDF — TWO-PHASE ORCHESTRATION
# ============================================================

def _checkpoint_is_done(checkpoint: PageCheckpoint) -> bool:
    if checkpoint.status not in COMPLETE_STATUSES:
        return False
    if checkpoint.status == "empty_page":
        return True  # no extraction file expected
    return bool(
        checkpoint.extraction_path and Path(checkpoint.extraction_path).exists()
    )


def process_pdf(pdf_path: Path):
    overall_start = time.perf_counter()

    (
        pdf_name,
        extraction_dir,
        raw_dir,
        checkpoint_dir,
        embedding_dir,
    ) = get_pdf_output_dirs(pdf_path)

    pdf_hash = pdf_identifier(pdf_path)

    print("\n" + "=" * 70)
    print(f"PDF: {pdf_path.name}")
    print("=" * 70)

    try:
        pdf = pdfium.PdfDocument(str(pdf_path))
    except Exception as e:
        print(f"✗ Cannot open PDF: {e}")
        return

    try:
        total_pages = len(pdf)
        print(f"Pages: {total_pages}")

        # Preload existing checkpoints (cheap, gives resume + full report).
        checkpoints: dict[int, PageCheckpoint] = {}
        for pn in range(1, total_pages + 1):
            ck = load_checkpoint(checkpoint_dir, pn)
            if ck is not None:
                checkpoints[pn] = ck

        # --------------------------------------------------------
        # PHASE 1 — NATIVE TEXT EXTRACTION (pypdfium2, all pages)
        # --------------------------------------------------------
        print("\n" + "-" * 70)
        print("PHASE 1 — NATIVE TEXT EXTRACTION (pypdfium2)")
        print("-" * 70)

        queue: list[tuple[int, str, list[str]]] = []
        prev_first_line: Optional[str] = None
        native_done = 0
        skipped = 0
        phase_start = time.perf_counter()

        for pn in range(START_PAGE, total_pages + 1):
            existing = checkpoints.get(pn)
            if (
                existing is not None
                and _checkpoint_is_done(existing)
                and not (REPROCESS_NATIVE and existing.method == "native")
            ):
                prev_first_line = existing.first_line
                skipped += 1
                continue

            t0 = time.perf_counter()
            result, native_text, reasons, first_line = try_native_page(
                pdf, pn, prev_first_line
            )
            elapsed = time.perf_counter() - t0
            prev_first_line = first_line

            if result is None:
                print(f"  → Page {pn}/{total_pages}: native insufficient "
                      f"({'; '.join(reasons)})")
                queue.append((pn, native_text, reasons))
                continue

            quality_status, quality_reasons = assess_quality(result)
            characters = calculate_extracted_characters(result)
            extraction_path = save_extraction(extraction_dir, result)

            checkpoint = PageCheckpoint(
                pdf_name=pdf_name,
                pdf_hash=pdf_hash,
                page_number=pn,
                status="success" if quality_status == "pass" else "review",
                method="native",
                attempts=1,
                successful_attempt=1,
                done_reason="native_text",
                time_seconds=round(elapsed, 3),
                model_time_seconds=0.0,
                sections=len(result.sections),
                blocks=len(result.blocks),
                tables=len(result.tables),
                figures=len(result.figures),
                extracted_characters=characters,
                quality_status=quality_status,
                quality_reasons=quality_reasons,
                extraction_path=str(extraction_path),
                first_line=first_line,
            )
            save_checkpoint(checkpoint_dir, checkpoint)
            checkpoints[pn] = checkpoint
            native_done += 1
            print(f"  ✓ Page {pn}/{total_pages}: native "
                  f"({characters:,} chars, {elapsed:.2f}s)")

        phase_time = time.perf_counter() - phase_start
        print(
            f"\n  Phase 1 done in {phase_time:.1f}s — "
            f"native: {native_done}, skipped (checkpointed): {skipped}, "
            f"queued for Qwen: {len(queue)}"
        )

        # --------------------------------------------------------
        # PHASE 2 — QWEN FALLBACK (only failed pages, model stays hot)
        # --------------------------------------------------------
        if queue:
            print("\n" + "-" * 70)
            print("PHASE 2 — QWEN2.5-VL FALLBACK (local model)")
            print("-" * 70)

            for pn, native_text, reasons in queue:
                print(f"\n  PAGE {pn}/{total_pages}")
                checkpoint = process_qwen_page(
                    pdf,
                    pdf_name,
                    pdf_hash,
                    pn,
                    total_pages,
                    extraction_dir,
                    raw_dir,
                    checkpoint_dir,
                    native_text=native_text,
                    native_reasons=reasons,
                )
                checkpoints[pn] = checkpoint
        else:
            print("  No pages need the vision model.")

    finally:
        pdf.close()

    # --------------------------------------------------------
    # PHASE 3 — EMBEDDINGS (batched)
    # --------------------------------------------------------
    embedding_result = process_embeddings(extraction_dir, embedding_dir)

    # --------------------------------------------------------
    # PHASE 4 — REPORT
    # --------------------------------------------------------
    processing_time = time.perf_counter() - overall_start
    page_results = [checkpoints[pn] for pn in sorted(checkpoints)]
    save_pdf_report(
        pdf_name,
        pdf_path,
        pdf_hash,
        total_pages,
        page_results,
        embedding_result,
        processing_time,
    )
    print(f"\n✓ Finished {pdf_path.name} in {processing_time:.1f}s")


# ============================================================
# EMBEDDINGS (batched, embeddinggemma template)
# ============================================================

def format_embedding_input(result: PageExtractionResult) -> tuple[str, bool]:
    body = build_embedding_text(result)
    truncated = len(body) > EMBED_MAX_CHARS
    body = body[:EMBED_MAX_CHARS]
    title = (result.title or "none").replace("\n", " ")
    # embeddinggemma expects: "title: ... | text: ..."
    return f"title: {title} | text: {body}", truncated


def _save_embedding_file(
    embedding_path: Path, page_number: int, embedding: list[float], truncated: bool
):
    data = {
        "page_number": page_number,
        "model": EMBED_MODEL,
        "dimension": len(embedding),
        "truncated": truncated,
        "embedding": embedding,
    }
    temp = embedding_path.with_suffix(".tmp")
    with open(temp, "w", encoding="utf-8") as f:
        json.dump(data, f)
    temp.replace(embedding_path)


def process_embeddings(extraction_dir: Path, embedding_dir: Path) -> dict:
    files = sorted(extraction_dir.glob("page_*.json"))

    print("\n" + "=" * 70)
    print("PHASE 3 — EMBEDDINGS")
    print("=" * 70)

    existing = 0
    successful = 0
    failed = 0

    pending: list[tuple[int, str, bool, Path]] = []

    for extraction_file in files:
        embedding_path = embedding_dir / extraction_file.name
        if embedding_path.exists():
            existing += 1
            continue
        try:
            with open(extraction_file, "r", encoding="utf-8") as f:
                result = PageExtractionResult.model_validate(json.load(f))
            text, truncated = format_embedding_input(result)
            if not text.strip():
                failed += 1
                print(f"  ✗ Empty text: {extraction_file.name}")
                continue
            pending.append((result.page_number, text, truncated, embedding_path))
        except Exception as e:
            failed += 1
            print(f"  ✗ Load failed {extraction_file.name}: {e}")

    # Batched calls: one network round-trip per EMBED_BATCH_SIZE pages.
    for i in range(0, len(pending), EMBED_BATCH_SIZE):
        batch = pending[i : i + EMBED_BATCH_SIZE]
        texts = [t for (_, t, _, _) in batch]

        embeddings: list[Optional[list[float]]] = []
        try:
            response = CLIENT.embed(
                model=EMBED_MODEL, input=texts, keep_alive=KEEP_ALIVE
            )
            embeddings = list(response.embeddings or [])
            if len(embeddings) != len(batch):
                raise RuntimeError("embedding count mismatch")
        except Exception as e:
            print(f"  ⚠ Batch embed failed ({e}) — falling back to per-page.")
            embeddings = []
            for (_, text, _, _) in batch:
                try:
                    r = CLIENT.embed(
                        model=EMBED_MODEL, input=[text], keep_alive=KEEP_ALIVE
                    )
                    embeddings.append((r.embeddings or [None])[0])
                except Exception:
                    embeddings.append(None)

        for (page_number, _, truncated, embedding_path), embedding in zip(
            batch, embeddings
        ):
            if not embedding:
                failed += 1
                print(f"  ✗ Embedding failed page {page_number}")
                continue
            _save_embedding_file(embedding_path, page_number, embedding, truncated)
            successful += 1

    if pending:
        print(f"  ✓ Embedded {successful} page(s), {failed} failed.")
    if existing:
        print(f"  ✓ {existing} embedding(s) already existed.")

    return {
        "successful": successful,
        "failed": failed,
        "existing": existing,
    }


# ============================================================
# PDF REPORT
# ============================================================

def save_pdf_report(
    pdf_name: str,
    pdf_path: Path,
    pdf_hash: str,
    total_pages: int,
    page_results: list[PageCheckpoint],
    embedding_result: dict,
    processing_time: float,
):
    successful = [x for x in page_results if x.status == "success"]
    review = [x for x in page_results if x.status == "review"]
    failed = [x for x in page_results if x.status == "needs_reextraction"]
    empty = [x for x in page_results if x.status == "empty_page"]
    native = [x for x in page_results if x.method == "native"]
    qwen = [x for x in page_results if x.method == "qwen"]
    recovered = [
        x for x in page_results if x.successful_attempt and x.successful_attempt > 1
    ]

    report = {
        "pdf": str(pdf_path),
        "pdf_name": pdf_name,
        "pdf_hash": pdf_hash,
        "total_pages": total_pages,
        "processed_pages": len(page_results),
        "successful_pages": len(successful),
        "review_pages": len(review),
        "needs_reextraction": len(failed),
        "empty_pages": len(empty),
        "native_pages": len(native),
        "qwen_pages": len(qwen),
        "recovered_by_retry": len(recovered),
        "success_rate": (
            len(successful) / total_pages if total_pages else 0
        ),
        "processing_time_seconds": round(processing_time, 2),
        "embedding": embedding_result,
        "reextraction_queue": [
            {
                "page": x.page_number,
                "method": x.method,
                "reason": x.error or "; ".join(x.quality_reasons),
                "attempts": x.attempts,
                "recommended_action": "reextract",
            }
            for x in failed
        ],
        "review_queue": [
            {
                "page": x.page_number,
                "method": x.method,
                "reason": "; ".join(x.quality_reasons),
                "recommended_action": "manual_review",
            }
            for x in review
        ],
        "pages": [x.model_dump() for x in page_results],
    }

    json_path = REPORTS_DIR / f"{pdf_name}_report.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)

    # Human readable
    txt_path = REPORTS_DIR / f"{pdf_name}_report.txt"
    lines = []
    lines.append("=" * 70)
    lines.append("MEDICAL PDF EXTRACTION REPORT")
    lines.append("=" * 70)
    lines.append(f"PDF: {pdf_path}")
    lines.append(f"Pages: {total_pages}")
    lines.append("")
    lines.append(f"Successful: {len(successful)}")
    lines.append(f"Review: {len(review)}")
    lines.append(f"Needs re-extraction: {len(failed)}")
    lines.append(f"Empty pages (terminal): {len(empty)}")
    lines.append(f"Native extraction: {len(native)}")
    lines.append(f"Qwen extraction: {len(qwen)}")
    lines.append(f"Recovered by retry: {len(recovered)}")
    lines.append(f"Success rate: {report['success_rate'] * 100:.2f}%")
    lines.append(f"Processing time: {processing_time:.2f}s")
    lines.append("")

    lines.append("RE-EXTRACTION QUEUE")
    lines.append("-" * 70)
    if not failed:
        lines.append("No pages require re-extraction.")
    else:
        for page in failed:
            reason = page.error or "; ".join(page.quality_reasons)
            lines.append(f"Page {page.page_number}: {reason}")

    lines.append("")
    lines.append("REVIEW QUEUE")
    lines.append("-" * 70)
    if not review:
        lines.append("No pages require review.")
    else:
        for page in review:
            lines.append(
                f"Page {page.page_number}: {'; '.join(page.quality_reasons)}"
            )

    lines.append("")
    lines.append("PAGE DETAILS")
    lines.append("-" * 70)
    for page in page_results:
        lines.append(
            f"Page {page.page_number}: {page.status} | "
            f"method={page.method} | "
            f"attempts={page.attempts} | "
            f"chars={page.extracted_characters} | "
            f"time={page.time_seconds:.2f}s"
        )

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    return json_path, txt_path


# ============================================================
# DETERMINE WHETHER PDF IS COMPLETE
# ============================================================

def pdf_is_complete(pdf_path: Path) -> bool:
    (
        pdf_name,
        extraction_dir,
        raw_dir,
        checkpoint_dir,
        embedding_dir,
    ) = get_pdf_output_dirs(pdf_path)

    try:
        pdf = pdfium.PdfDocument(str(pdf_path))
        total_pages = len(pdf)
        pdf.close()
    except Exception:
        return False

    for page_number in range(1, total_pages + 1):
        checkpoint = load_checkpoint(checkpoint_dir, page_number)
        if checkpoint is None:
            return False
        if not _checkpoint_is_done(checkpoint):
            return False

    return True


# ============================================================
# SELECT NEXT PDF
# ============================================================

def select_next_pdf(exclude: set[Path]) -> Optional[Path]:
    try:
        pdf_files = sorted(
            path
            for path in INPUT_DIR.iterdir()
            if path.is_file() and path.suffix.lower() in PDF_EXTENSIONS
        )
    except FileNotFoundError:
        return None

    for pdf_path in pdf_files:
        if pdf_path in exclude:
            continue
        if pdf_is_complete(pdf_path):
            print(f"✓ Already complete: {pdf_path.name}")
            continue
        return pdf_path
    return None


# ============================================================
# MAIN
# ============================================================

def main():
    if not test_ollama_connection():
        return

    attempted: set[Path] = set()

    while True:
        pdf_path = select_next_pdf(exclude=attempted)
        if pdf_path is None:
            print("\n✓ All PDFs processed.")
            break

        attempted.add(pdf_path)
        try:
            process_pdf(pdf_path)
        except Exception as e:
            print(f"\n✗ PDF failed: {type(e).__name__}: {e}")
            traceback.print_exc()


if __name__ == "__main__":
    main()


OLLAMA CONNECTION

Installed models:
  - qwen2.5vl:3b
  - embeddinggemma:latest

✓ Ollama is reachable.
✓ Required models are installed.

PDF: dc24s010.pdf
Pages: 40

----------------------------------------------------------------------
PHASE 1 — NATIVE TEXT EXTRACTION (pypdfium2)
----------------------------------------------------------------------
  ✓ Page 22/40: native (2,764 chars, 0.05s)
  ✓ Page 29/40: native (7,354 chars, 0.05s)
  ✓ Page 36/40: native (9,375 chars, 0.13s)
  ✓ Page 37/40: native (9,117 chars, 0.10s)
  ✓ Page 38/40: native (9,351 chars, 0.10s)
  ✓ Page 39/40: native (9,323 chars, 0.07s)
  ✓ Page 40/40: native (1,671 chars, 0.02s)

  Phase 1 done in 0.7s — native: 7, skipped (checkpointed): 33, queued for Qwen: 0
  No pages need the vision model.

PHASE 3 — EMBEDDINGS
  ✓ Embedded 40 page(s), 0 failed.

✓ Finished dc24s010.pdf in 56.4s

PDF: dc26s005.pdf
Pages: 43

----------------------------------------------------------------------
PHASE 1 — NATIVE TEXT EXTRA